# Compute Engine 인스턴스 생성 및 Ops Agent 정책 구성

Google Cloud Platform(GCP)에서 Compute Engine VM 인스턴스를 프로비저닝하고, 리전별 비용 비교, Ops Agent 정책 적용 및 과금 방지 점검을 수행하는 노트북입니다.

## 1. 💰 리전별 비용 분석 및 최저가 리전 TOP 3 확인
현재 인스턴스 옵션(`e2-medium` [vCPU 2, RAM 4GB] + `pd-balanced` 10GB 부팅 디스크) 기준, 전 세계 GCP 리전 중 가장 저렴한 리전들을 비교 분석합니다.

In [2]:
!pip

Package                 Version
----------------------- -----------
annotated-doc           0.0.5
annotated-types         0.8.0
anyio                   4.15.1
asttokens               3.0.2
certifi                 2026.7.22
cffi                    2.1.1
charset-normalizer      3.5.1
click                   8.5.0
colorama                0.4.6
comm                    0.2.3
cryptography            50.0.1
debugpy                 1.8.21
distro                  1.9.0
executing               2.2.1
fastapi                 0.141.1
google-auth             2.57.1
google-genai            2.22.0
h11                     0.16.0
httpcore                1.0.9
httpx                   0.28.1
idna                    3.19
ipykernel               7.3.0
ipython                 9.17.1
ipython_pygments_lexers 1.1.1
jedi                    0.20.0
jupyter_client          8.10.0
jupyter_core            5.9.1
markdown-it-py          4.2.0
matplotlib-inline       0.2.2
mdurl                   0.1.2
nest-asyncio2    

In [3]:
!pip install pandas

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ----------------- ---------------------- 4.2/9.8 MB 22.9 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.8 MB 22.9 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 21.9 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------- ----------------------------- 3.4/12.6 MB 16.8 MB/s eta 0:00:01
   ------------------------- -------------- 7.9/12.6 MB 18.7 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 19.8 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 19.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- ----------------------

In [4]:
# 리전별 e2-medium + pd-balanced 10GB 비용 비교 데이터프레임
import pandas as pd

pricing_data = [
    {"순위": "1위 (공동)", "리전 ID": "us-central1", "지역": "미국 중부 (아이오와)", "추천 Zone": "us-central1-a", "시간당 비용": "$0.03488 / hr", "월간 비용": "$25.46 / mo", "비고": "현재 사용 리전 (최저가군)"},
    {"순위": "1위 (공동)", "리전 ID": "us-east5", "지역": "미국 동부 (콜럼버스)", "추천 Zone": "us-east5-a", "시간당 비용": "$0.03488 / hr", "월간 비용": "$25.46 / mo", "비고": "미국 동부 최저가군"},
    {"순위": "1위 (공동)", "리전 ID": "us-east1 / us-west1", "지역": "미국 사우스캐롤라이나/오리건", "추천 Zone": "us-east1-b", "시간당 비용": "$0.03488 / hr", "월간 비용": "$25.46 / mo", "비고": "미국 Tier 1 표준 최저가군"},
    {"순위": "4위", "리전 ID": "europe-north2", "지역": "유럽 북부", "추천 Zone": "europe-north2-a", "시간당 비용": "$0.03655 / hr", "월간 비용": "$26.68 / mo", "비고": "유럽 권역 최저가 (+4.8%)"},
    {"순위": "5위", "리전 ID": "europe-west1", "지역": "유럽 서부 (벨기에)", "추천 Zone": "europe-west1-b", "시간당 비용": "$0.03823 / hr", "월간 비용": "$27.91 / mo", "비고": "유럽 서부 최저가 (+9.6%)"},
    {"순위": "참고", "리전 ID": "asia-northeast3", "지역": "아시아 동북부 (서울)", "추천 Zone": "asia-northeast3-a", "시간당 비용": "$0.04353 / hr", "월간 비용": "$31.78 / mo", "비고": "서울 리전 (최저가 대비 +24.8%)"}
]

df_pricing = pd.DataFrame(pricing_data)
print("=== [e2-medium + pd-balanced 10GB] 리전별 예상 비용 비교 TOP 3 ===")
display(df_pricing)

=== [e2-medium + pd-balanced 10GB] 리전별 예상 비용 비교 TOP 3 ===


,순위,리전 ID,지역,추천 Zone,시간당 비용,월간 비용,비고
0,1위 (공동),us-central1,미국 중부 (아이오와),us-central1-a,$0.03488 / hr,$25.46 / mo,현재 사용 리전 (최저가군)
1,1위 (공동),us-east5,미국 동부 (콜럼버스),us-east5-a,$0.03488 / hr,$25.46 / mo,미국 동부 최저가군
2,1위 (공동),us-east1 / us-west1,미국 사우스캐롤라이나/오리건,us-east1-b,$0.03488 / hr,$25.46 / mo,미국 Tier 1 표준 최저가군
3,4위,europe-north2,유럽 북부,europe-north2-a,$0.03655 / hr,$26.68 / mo,유럽 권역 최저가 (+4.8%)
4,5위,europe-west1,유럽 서부 (벨기에),europe-west1-b,$0.03823 / hr,$27.91 / mo,유럽 서부 최저가 (+9.6%)
5,참고,asia-northeast3,아시아 동북부 (서울),asia-northeast3-a,$0.04353 / hr,$31.78 / mo,서울 리전 (최저가 대비 +24.8%)


### (심화) GCP Cloud Billing Catalog API를 통한 실시간 가격 조회 코드
GCP의 공식 과금 카탈로그 API에서 최신 SKU 가격을 실시간으로 쿼리하여 리전별 요금을 집계합니다.

In [5]:
import subprocess, urllib.request, json

def get_gcp_access_token():
    return subprocess.check_output('gcloud auth print-access-token', shell=True).decode().strip()

def fetch_cheapest_regions():
    token = get_gcp_access_token()
    base_url = "https://cloudbilling.googleapis.com/v1/services/6F81-5844-456A/skus"
    core_prices, ram_prices, disk_prices = {}, {}, {}
    page_token = ""
    
    print("Cloud Billing Catalog API에서 가격 데이터를 조회 중...")
    while True:
        url = f"{base_url}?pageSize=500" + (f"&pageToken={page_token}" if page_token else "")
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        with urllib.request.urlopen(req) as resp:
            data = json.loads(resp.read().decode())
            for s in data.get("skus", []):
                desc = s.get("description", "")
                regions = s.get("serviceRegions", [])
                if not regions or any(x in desc for x in ["Spot", "Preemptible", "Commitment", "Committed", "Custom"]):
                    continue
                region = regions[0]
                pricing_info = s.get("pricingInfo", [])
                if not pricing_info:
                    continue
                tiered_rates = pricing_info[0].get("pricingExpression", {}).get("tieredRates", [])
                if not tiered_rates:
                    continue
                rates = tiered_rates[0].get("unitPrice", {})
                price = int(rates.get("units", 0)) + int(rates.get("nanos", 0)) / 1e9
                
                if "E2 Instance Core running in " in desc:
                    core_prices[region] = price
                elif "E2 Instance Ram running in " in desc:
                    ram_prices[region] = price
                elif "Balanced PD Capacity" in desc and "Regional" not in desc:
                    disk_prices[region] = price
            
            page_token = data.get("nextPageToken")
            if not page_token:
                break
                
    results = []
    for reg in core_prices:
        if reg in ram_prices:
            hourly = (core_prices[reg] * 1.0) + (ram_prices[reg] * 4.0) + ((disk_prices.get(reg, 0.10) * 10.0) / 730.0)
            results.append({"리전": reg, "시간당 비용($)": round(hourly, 5), "월 예상 비용($)": round(hourly * 730, 2)})
            
    results.sort(key=lambda x: x["시간당 비용($)"])
    return pd.DataFrame(results[:5])

# 실시간 API 데이터 기반 TOP 5 리전 출력
top_regions_df = fetch_cheapest_regions()
display(top_regions_df)

Cloud Billing Catalog API에서 가격 데이터를 조회 중...


,리전,시간당 비용($),월 예상 비용($)
0,us-east5,0.03488,25.46
1,us-central1,0.03488,25.46
2,us-west8,0.03488,25.46
3,europe-north2,0.03655,26.68
4,northamerica-south1,0.03801,27.75


---
## 2. 일괄 실행 (인스턴스 생성 + Ops Agent 정책 등록)
가장 저렴한 리전인 `us-central1`(`us-central1-a`)을 기준으로 인스턴스 생성 및 설정을 한 번에 실행합니다.

In [6]:
!gcloud compute instances create instance-20260914-055119 \
    --project=iceu-songpa30 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=584903808975-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-055119,disk-resource-policy=projects/iceu-songpa30/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any \
&& \
printf 'agentsRule:\n  packageState: installed\n  version: latest\ninstanceFilter:\n  inclusionLabels:\n  - labels:\n      goog-ops-agent-policy: v2-template-1-7-0\n' > config.yaml \
&& \
gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa30 \
    --zone=us-central1-a \
    --file=config.yaml

=== [e2-medium + pd-balanced 10GB] 리전별 예상 비용 비교 TOP 3 ===


,순위,리전 ID,지역,추천 Zone,시간당 비용,월간 비용,비고
0,1위 (공동),us-central1,미국 중부 (아이오와),us-central1-a,$0.03488 / hr,$25.46 / mo,현재 사용 리전 (최저가군)
1,1위 (공동),us-east5,미국 동부 (콜럼버스),us-east5-a,$0.03488 / hr,$25.46 / mo,미국 동부 최저가군
2,1위 (공동),us-east1 / us-west1,미국 사우스캐롤라이나/오리건,us-east1-b,$0.03488 / hr,$25.46 / mo,미국 Tier 1 표준 최저가군
3,4위,europe-north2,유럽 북부,europe-north2-a,$0.03655 / hr,$26.68 / mo,유럽 권역 최저가 (+4.8%)
4,5위,europe-west1,유럽 서부 (벨기에),europe-west1-b,$0.03823 / hr,$27.91 / mo,유럽 서부 최저가 (+9.6%)
5,참고,asia-northeast3,아시아 동북부 (서울),asia-northeast3-a,$0.04353 / hr,$31.78 / mo,서울 리전 (최저가 대비 +24.8%)


---
## 3. 단계별 실행 (권장: 디버깅 및 세부 확인 용이)
각 단계를 개별 셀로 나누어 순서대로 실행하고 결과를 확인할 수 있습니다.

### Step 1: Compute Engine 인스턴스 생성

In [ ]:
!gcloud compute instances create instance-20260915-055119 \
    --project=iceu-songpa30 \
    --zone=us-central1-a \
    --machine-type=e2-medium \
    --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default \
    --metadata=enable-osconfig=TRUE \
    --maintenance-policy=MIGRATE \
    --provisioning-model=STANDARD \
    --service-account=584903808975-compute@developer.gserviceaccount.com \
    --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
    --create-disk=auto-delete=yes,boot=yes,device-name=instance-20260914-055119,disk-resource-policy=projects/iceu-songpa30/regions/us-central1/resourcePolicies/default-schedule-1,image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced \
    --no-shielded-secure-boot \
    --shielded-vtpm \
    --shielded-integrity-monitoring \
    --labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud \
    --reservation-affinity=any

NAME                      ZONE           MACHINE_TYPE  PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
instance-20260914-055119  us-central1-a  e2-medium                  10.128.0.3   34.57.178.39  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/iceu-songpa30/zones/us-central1-a/instances/instance-20260914-055119].


### Step 2: Ops Agent 정책 설정 파일 (`config.yaml`) 생성

In [8]:
config_content = """agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
"""

with open("config.yaml", "w", encoding="utf-8") as f:
    f.write(config_content)

print("config.yaml 파일이 생성되었습니다.")

config.yaml 파일이 생성되었습니다.


### Step 3: Ops Agent 정책 등록

In [9]:
!gcloud compute instances ops-agents policies create goog-ops-agent-v2-template-1-7-0-us-central1-a \
    --project=iceu-songpa30 \
    --zone=us-central1-a \
    --file=config.yaml

ERROR: (gcloud.compute.instances.ops-agents.policies.create) ALREADY_EXISTS: Requested entity already exists


In [ ]:
# 현재 등록된 정책 목록 확인
!gcloud compute instances ops-agents policies list --zone=us-central1-a


Listed 0 items.


---
## 4. 리소스 정리 (인스턴스 및 Ops Agent 정책 삭제)


In [12]:
# 1. Compute Engine 인스턴스 삭제
!gcloud compute instances delete instance-20260914-055119 --zone=us-central1-a --quiet

# 2. Ops Agent 정책 삭제
!gcloud compute instances ops-agents policies delete goog-ops-agent-v2-template-1-7-0-us-central1-a --zone=us-central1-a --quiet


ERROR: (gcloud.compute.instances.delete) Could not fetch resource:
 - The resource 'projects/iceu-songpa30/zones/us-central1-a/instances/instance-20260914-055119' was not found

ERROR: (gcloud.compute.instances.ops-agents.policies.delete) Encountered a malformed Cloud Ops Agents Policy.
 The Cloud Ops Agents policy [goog-ops-agent-v2-template-1-7-0-us-central1-a] may have been modified directly by the OS Config API / gcloud commands. If so, please delete and re-create with the Ops Agents policy gcloud commands. If not, this may be an internal error.


### 인스턴스 및 Ops Agent 정책 삭제 여부 확인
삭제 명령 실행 후 잔여 인스턴스와 Ops Agent 정책이 없는지 조회합니다.

In [13]:
# 남아있는 인스턴스 목록 확인
print('=== 1. 잔여 Compute Engine 인스턴스 확인 ===')
!gcloud compute instances list

# 남아있는 Ops Agent 정책 확인
print('\n=== 2. 잔여 Ops Agent 정책 확인 ===')
!gcloud compute instances ops-agents policies list --zone=us-central1-a


=== 1. 잔여 Compute Engine 인스턴스 확인 ===

=== 2. 잔여 Ops Agent 정책 확인 ===


Listed 0 items.
Listed 0 items.


---
## 5. 🔍 과금 방지 하네스: 잔여 리소스 및 과금 위험 점검
인스턴스 삭제 후 남아있는 디스크, 미사용 고정 IP, 스냅샷 등 하룻밤 사이 과금을 유발할 수 있는 리소스가 있는지 종합 검사합니다.

In [14]:
# 잔여 리소스 점검 스크립트 실행
!python check_gcp_resources.py

🔍 [GCP 과금 방지 하네스] 잔여 리소스 및 과금 위험 점검 중...
✅ [Compute VM 인스턴스] 0개 (정상/과금 위험 없음)
✅ [영구 디스크 (Persistent Disks)] 0개 (정상/과금 위험 없음)
✅ [정적 외부 IP (Static External IPs)] 0개 (정상/과금 위험 없음)
✅ [디스크 스냅샷 (Snapshots)] 0개 (정상/과금 위험 없음)
✅ [부하분산 포워딩 규칙 (Load Balancers)] 0개 (정상/과금 위험 없음)

🎉 [안전] 남아있는 과금 대상 리소스가 없습니다!
   모든 핵심 리소스가 정상 정리되어 안심하고 다음 날로 넘어가셔도 좋습니다.
